# OLS Distribution (iid) and Some Diagnostics

This notebook studies key properties of the model (distribution under iid assumptions, measures of fit, normality, multicollinearity).

You may also consider the [HypothesisTests.jl](https://github.com/JuliaStats/HypothesisTests.jl) package (not used here).

## Load Packages and Extra Functions

The key functions for the diagnostic tests are from the (local) `FinEcmt_OLS` module.

In [1]:
MyModulePath = joinpath(pwd(),"src")
!in(MyModulePath,LOAD_PATH) && push!(LOAD_PATH,MyModulePath)
using FinEcmt_OLS

In [2]:
#=
include(joinpath(pwd(),"src","FinEcmt_OLS.jl"))
using .FinEcmt_OLS
=#

In [3]:
using DelimitedFiles, Statistics, LinearAlgebra

## Loading Data

In [4]:
x = readdlm("Data/TwoIndustries.csv",',',skipstart=1)

(dN,Re,F) = (x[:,1],Float64.(x[:,2:3]),Float64.(x[:,4:end]))   #make sure data is Float
x = nothing
printlnPs("Re: ",size(Re),"\n","F: ",size(F))

y = Re[:,1]                    #to get standard OLS notation, here one return series
T = size(y,1)
x = [ones(T) F]
k = size(x,2)

println("\nT and k: $T $k")

      Re:   (660, 2)          
       F:   (660, 3)

T and k: 660 4


In [5]:
(b,u,_,V,R²) = OlsGM(y,x)    #do OLS
Stdb = sqrt.(diag(V))

printblue("OLS with traditional standard errors:\n")
xNames = ["c","market","SMB","HML"]
printmat([b Stdb],colNames=["coef","std"],rowNames=xNames)

OLS with traditional standard errors:

            coef       std
c          0.142     0.109
market     1.130     0.025
SMB        0.180     0.036
HML       -0.510     0.036



In [6]:
σ² = var(u;corrected=false)  #variance of (fitted) residual, perhaps drop `;corrected=false`
Sxx = x'*x
V_alt = inv(Sxx)*σ²    #just double checking the VCV 

Stdb_alt = sqrt.(diag(V_alt))
printmat(Stdb,Stdb_alt;colNames=["Std (earlier)","Std (new)"],rowNames=xNames)

      Std (earlier) Std (new)
c          0.109     0.109
market     0.025     0.025
SMB        0.036     0.036
HML        0.036     0.036



# Measures of Fit

Adjusted R², AIC, BIC  (the two latter is discussed in more detail in another chapter)

In [7]:
@doc2 RegressionFit
#using CodeTracking
#println(@code_string RegressionFit([1],0.0,3))    #print the source code

```julia
RegressionFit(u,R²,k)
```

Calculate adjusted R², AIC and BIC from regression residuals.

### Input

  * `u::Vector`:      T-vector of residuals
  * `R²::Float`:      the R² value
  * `k::Int`:         number of regressors


In [8]:
(R²adj,AIC,BIC) = RegressionFit(u,R²,k)

printblue("Measures of fit")
printmat([R²,R²adj,AIC,BIC];rowNames=["R²","R²adj","AIC","BIC"])

Measures of fit
R²        0.823
R²adj     0.822
AIC       2.035
BIC       2.062



# Test of Normality

of the residuals, applying the Jarque-Bera test.

In [9]:
@doc2 JarqueBeraTest
#println(@code_string JarqueBeraTest([1]))    #print the source code

```julia
JarqueBeraTest(x)
```

Calculate the JB test for each column in a matrix. Reports `(skewness,kurtosis,JB)`.


In [10]:
(skewness,kurtosis,JB,pvals) = JarqueBeraTest(u)

printblue("Test of normality")
xut = vcat(skewness,kurtosis,JB)
printmat(xut,collect(pvals);rowNames=["skewness","kurtosis","Jarque-Bera"],colNames=["stat","p-value"])

Test of normality
                 stat   p-value
skewness        0.119     0.211
kurtosis        3.895     0.000
Jarque-Bera    23.598     0.000



# Multicollinearity

by studying the correlation matrix and the variance inflation factor (VIF). A high VIF (5 to 10) might indicate issues with multicollinearity.

In [11]:
@doc2 VIF
#println(@code_string VIF([1]))    #print the source code

```julia
VIF(X)
```

Calculate the variance inflation factor

### Input

  * `x::Matrix`:    Txk matrix with regressors

### Output

  * `maxVIF::Float`:     highest VIF value
  * `allVIF::Vector`:    a k VIF values


In [12]:
printblue("Correlation matrix (checking multicollinearity)")
printmat(cor(x);colNames=xNames,rowNames=xNames)

Correlation matrix (checking multicollinearity)
               c    market       SMB       HML
c          1.000       NaN       NaN       NaN
market       NaN     1.000     0.281    -0.213
SMB          NaN     0.281     1.000    -0.155
HML          NaN    -0.213    -0.155     1.000



In [13]:
(maxVIF,allVIF) = VIF(x)
printblue("VIF (checking multicollinearity)")
printmat(allVIF;rowNames=xNames)

VIF (checking multicollinearity)
c          1.000
market     1.121
SMB        1.097
HML        1.058



# A Convenience Function for Printing All These Tests (extra)

In [14]:
@doc2 DiagnosticsTable
#println(@code_string DiagnosticsTable([1],[1],0.0))    #print the source code

```julia
DiagnosticsTable(X,u,R²,nlags,xNames="")
```

Compute and print a number of regression diagnostic tests.

### Input

  * `X::Matrix`:      Txk matrix of regressors
  * `u::Vector`:      T-vector of residuals
  * `R²::Float`:      the R² value
  * `xNames::Vector`: of strings, regressor names


In [15]:
DiagnosticsTable(x,u,R²,xNames)

Test of all slopes = 0
stat   3072.554
p-val     0.000

Measures of fit
R²        0.823
R²adj     0.822
AIC       2.035
BIC       2.062

Test of normality
                 stat   p-value
skewness        0.119     0.211
kurtosis        3.895     0.000
Jarque-Bera    23.598     0.000

Correlation matrix (checking multicollinearity)
               c    market       SMB       HML
c          1.000       NaN       NaN       NaN
market       NaN     1.000     0.281    -0.213
SMB          NaN     0.281     1.000    -0.155
HML          NaN    -0.213    -0.155     1.000

VIF (checking multicollinearity)
c          1.000
market     1.121
SMB        1.097
HML        1.058

